# 🚀 Entrenamiento de IA: Generador de Prompts Hiperrealistas para Google Nano Banana Pro
### Método: Fine-Tuning con DPO (Direct Preference Optimization) usando Unsloth y QLoRA en GPU T4 Gratuita

Este cuaderno entrena un modelo de lenguaje compacto (`Qwen-2.5-3B-Instruct` o `Llama-3.2-3B-Instruct`) para que transforme cualquier idea simple en un **prompt fotográfico ultra realista** que engañe al ojo humano y elimine por completo el aspecto de IA en Google Nano Banana Pro.

## Paso 1: Instalación de Unsloth y librerías optimizadas
Unsloth acelera el entrenamiento hasta 5x y reduce el consumo de VRAM en un 80%, permitiendo entrenar en la GPU gratuita de Colab sin agotar memoria.

In [ ]:
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes datasets

## Paso 2: Cargar el Modelo Base con Cuantización de 4 bits
Utilizaremos `Qwen/Qwen2.5-3B-Instruct-bnb-4bit` (o Llama-3.2), ideal por su excelente razonamiento en español e inglés y su bajo consumo de memoria (< 3.5 GB de VRAM).

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 1024  # Suficiente para prompts extensos y detallados
dtype = None  # Detección automática (Float16 en T4, Bfloat16 en A100)
load_in_4bit = True  # Reduce uso de memoria a < 4GB

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-3B-Instruct-bnb-4bit",
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

print("✅ Modelo base cargado exitosamente en la GPU.")

## Paso 3: Configurar Adaptadores LoRA / QLoRA
Solo entrenamos matrices de bajo rango (`rank = 16`), manteniendo los pesos base congelados. Esto garantiza un entrenamiento rápido y evita el olvido catastrófico.

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

print("✅ Adaptadores LoRA configurados listos para entrenamiento.")

## Paso 4: Cargar el Dataset DPO (`dataset_dpo_nano_banana.jsonl`)
Ejecuta la celda para subir el archivo `dataset_dpo_nano_banana.jsonl` generado en la Fase 2, o crearlo automáticamente si no lo subes manualmente.

In [ ]:
import os
import json
from google.colab import files
from datasets import Dataset

if not os.path.exists("dataset_dpo_nano_banana.jsonl"):
    print("Por favor, sube el archivo 'dataset_dpo_nano_banana.jsonl':")
    uploaded = files.upload()

# Cargar y formatear el dataset para DPO
data_list = []
with open("dataset_dpo_nano_banana.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        item = json.loads(line)
        data_list.append({
            "prompt": item["prompt"],
            "chosen": item["chosen"],
            "rejected": item["rejected"]
        })

dataset = Dataset.from_list(data_list)
print(f"✅ Dataset cargado: {len(dataset)} pares de entrenamiento DPO.")
print("Ejemplo:\nPrompt:", dataset[0]['prompt'][:80], "...\nChosen:", dataset[0]['chosen'][:100], "...")

## Paso 5: Configurar y Ejecutar el Entrenamiento DPO (`DPOTrainer`)
DPO enseña matemáticamente a la IA a maximizar la probabilidad del estilo fotográfico analógico (`chosen`) y penalizar el estilo artificial de IA con palabras vacías (`rejected`).

In [ ]:
from trl import DPOTrainer, DPOConfig
from unsloth import is_bfloat16_supported

dpo_config = DPOConfig(
    output_dir="./resultados_dpo_nano_banana",
    beta=0.1,  # Parámetro de temperatura de DPO (controla la fuerza del castigo a clichés de IA)
    learning_rate=5e-5,
    lr_scheduler_type="cosine",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    warmup_ratio=0.1,
    fp16=not is_bfloat16_supported(),
    bf16=is_bfloat16_supported(),
    logging_steps=5,
    save_strategy="no",
    seed=3407,
)

dpo_trainer = DPOTrainer(
    model=model,
    ref_model=None,  # Unsloth maneja el modelo de referencia sin duplicar memoria
    args=dpo_config,
    train_dataset=dataset,
    tokenizer=tokenizer,
    max_length=max_seq_length,
    max_prompt_length=256,
)

print("🔥 Iniciando entrenamiento DPO... Toma aprox. 10 a 15 minutos en GPU T4.")
trainer_stats = dpo_trainer.train()
print("🎉 ¡Entrenamiento completado exitosamente!")

## Paso 6: Prueba en Vivo de Inferencia
Probamos el modelo entrenado con una idea nueva y verificamos que aplique las 7 capas fotográficas y elimine el aspecto de IA.

In [ ]:
FastLanguageModel.for_inference(model)  # Optimiza para generación rápida 2x

def generar_prompt_anti_ia(idea_usuario):
    prompt_formateado = f"Generate a hyperrealistic, non-AI looking photograph prompt for: {idea_usuario}"
    inputs = tokenizer([prompt_formateado], return_tensors="pt").to("cuda")
    
    outputs = model.generate(
        **inputs,
        max_new_tokens=250,
        temperature=0.7,
        top_p=0.9,
        repetition_penalty=1.1
    )
    resultado = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # Limpiar el prefijo de entrada
    if prompt_formateado in resultado:
        resultado = resultado.replace(prompt_formateado, "").strip()
    return resultado

# Prueba con un caso nuevo
idea_test = "Mujer de 45 años cocinando en su cocina de campo en un día soleado"
prompt_final = generar_prompt_anti_ia(idea_test)

print("=== IDEA ORIGINAL ===")
print(idea_test)
print("\n=== PROMPT MAESTRO GENERADO PARA GOOGLE NANO BANANA PRO ===")
print(prompt_final)

## Paso 7: Guardar y Exportar el Modelo
Puedes guardar el adaptador LoRA (ocupa apenas ~30 MB) o exportar a formato GGUF para usarlo localmente con Ollama o LM Studio.

In [ ]:
# Guardar adaptador LoRA
model.save_pretrained_merged("lora_dpo_nano_banana", tokenizer, save_method="lora")
print("✅ Adaptador LoRA guardado en carpeta 'lora_dpo_nano_banana'.")

# Para comprimir y descargar a tu computadora:
!zip -r lora_dpo_nano_banana.zip lora_dpo_nano_banana
files.download("lora_dpo_nano_banana.zip")
print("💾 Descarga iniciada.")